In [3]:

import pandas as pd
import numpy as np
from scipy import stats
import os

# Create output directory
os.makedirs("outputs", exist_ok=True)

# Load dataset
df = pd.read_csv("Sample - Superstore.csv", encoding="cp1252")

# Basic data cleaning
df = df.dropna(subset=[
    "Order ID", "Segment", "Sales", "Profit"
])

# Convert numeric columns
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")
df = df.dropna(subset=["Sales", "Profit"])

# Aggregate line items into order-level data
orders = (
    df.groupby(["Order ID", "Segment"], as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum")
      )
)

# Function to calculate 95% confidence interval
def confidence_interval(data, confidence=0.95):
    data = np.asarray(data)
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)

    margin = stats.t.ppf(
        (1 + confidence) / 2,
        df=n - 1
    ) * sem

    return mean - margin, mean + margin


# Interpret p-value
def interpret_pvalue(p_value):
    if p_value < 0.05:
        return "Reject H0: Statistically significant result"
    return "Fail to reject H0: Insufficient statistical evidence"


print("\nHYPOTHESIS TESTING ANALYSIS")
print("=" * 55)

# ------------------------------------------------
# SCENARIO 1: ONE-SAMPLE T-TEST (SALES BENCHMARK)
# ------------------------------------------------

sales = orders["Sales"]
benchmark = 200

t_stat, p_value = stats.ttest_1samp(
    sales,
    popmean=benchmark
)

ci_low, ci_high = confidence_interval(sales)

print("\nSCENARIO 1: Average Sales vs $200")
print("H0: Average order sales = $200")
print("H1: Average order sales != $200")
print(f"Sample Mean: ${sales.mean():.2f}")
print(f"Benchmark: ${benchmark:.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"95% CI: (${ci_low:.2f}, ${ci_high:.2f})")
print(interpret_pvalue(p_value))


# ------------------------------------------------
# SCENARIO 2: TWO-SAMPLE T-TEST
# CONSUMER vs CORPORATE
# ------------------------------------------------

consumer = orders.loc[
    orders["Segment"] == "Consumer", "Sales"
]

corporate = orders.loc[
    orders["Segment"] == "Corporate", "Sales"
]

t_stat, p_value = stats.ttest_ind(
    consumer,
    corporate,
    equal_var=False
)

diff = consumer.mean() - corporate.mean()

# Welch confidence interval for difference in means
se = np.sqrt(
    consumer.var(ddof=1) / len(consumer)
    + corporate.var(ddof=1) / len(corporate)
)

df_welch = se**4 / (
    (consumer.var(ddof=1) / len(consumer))**2
    / (len(consumer) - 1)
    +
    (corporate.var(ddof=1) / len(corporate))**2
    / (len(corporate) - 1)
)

margin = stats.t.ppf(0.975, df_welch) * se

print("\nSCENARIO 2: Consumer vs Corporate Sales")
print("H0: Mean sales are equal")
print("H1: Mean sales are different")
print(f"Consumer Mean: ${consumer.mean():.2f}")
print(f"Corporate Mean: ${corporate.mean():.2f}")
print(f"Mean Difference: ${diff:.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(
    f"95% CI for difference: "
    f"(${diff - margin:.2f}, ${diff + margin:.2f})"
)
print(interpret_pvalue(p_value))


# ------------------------------------------------
# SCENARIO 3: ONE-SAMPLE T-TEST (PROFITABILITY)
# ------------------------------------------------

profit = orders["Profit"]

t_stat, p_value = stats.ttest_1samp(
    profit,
    popmean=0
)

ci_low, ci_high = confidence_interval(profit)

print("\nSCENARIO 3: Average Order Profitability")
print("H0: Average order profit = $0")
print("H1: Average order profit != $0")
print(f"Average Profit: ${profit.mean():.2f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"95% CI: (${ci_low:.2f}, ${ci_high:.2f})")
print(interpret_pvalue(p_value))


# ------------------------------------------------
# TYPE I AND TYPE II ERRORS
# ------------------------------------------------

print("\nTYPE I AND TYPE II ERRORS")
print("-" * 55)

print("""
Type I Error:
Rejecting a true null hypothesis.
Business consequence:
A company may believe customer segments
have different sales when they actually do not.

Type II Error:
Failing to reject a false null hypothesis.
Business consequence:
A company may miss a real difference in
customer spending and lose an opportunity
to improve its marketing strategy.
""")

print("\nAnalysis completed successfully.")



HYPOTHESIS TESTING ANALYSIS

SCENARIO 1: Average Sales vs $200
H0: Average order sales = $200
H1: Average order sales != $200
Sample Mean: $458.61
Benchmark: $200.00
T-statistic: 19.1711
P-value: 0.000000
95% CI: ($432.17, $485.06)
Reject H0: Statistically significant result

SCENARIO 2: Consumer vs Corporate Sales
H0: Mean sales are equal
H1: Mean sales are different
Consumer Mean: $449.11
Corporate Mean: $466.41
Mean Difference: $-17.30
T-statistic: -0.5895
P-value: 0.555584
95% CI for difference: ($-74.84, $40.24)
Fail to reject H0: Insufficient statistical evidence

SCENARIO 3: Average Order Profitability
H0: Average order profit = $0
H1: Average order profit != $0
Average Profit: $57.18
T-statistic: 11.7315
P-value: 0.000000
95% CI: ($47.62, $66.73)
Reject H0: Statistically significant result

TYPE I AND TYPE II ERRORS
-------------------------------------------------------

Type I Error:
Rejecting a true null hypothesis.
Business consequence:
A company may believe customer segme